In [ ]:
# CELL 1: Install
%%capture install_out
!pip install vllm aiohttp nest_asyncio tenacity pydantic

import subprocess
print("Install done.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

In [ ]:
# CELL 2: Upload DB file + fix Colab async
import nest_asyncio
nest_asyncio.apply()

from google.colab import files
print("Upload your .db file:")
uploaded = files.upload()

# After upload, find the DB path
import os
db_files = [f for f in uploaded.keys() if f.endswith('.db')]
DB_PATH = f"/content/{db_files[0]}"
print(f"DB loaded at: {DB_PATH}")

# Quick verify
import sqlite3
conn = sqlite3.connect(DB_PATH)
q_count = conn.execute("SELECT COUNT(*) FROM queries").fetchone()[0]
c_count = conn.execute("SELECT COUNT(*) FROM claims").fetchone()[0]
ao_count = conn.execute("SELECT COUNT(*) FROM agent_outputs").fetchone()[0]
conn.close()
print(f"queries={q_count}  claims={c_count}  agent_outputs={ao_count} (should be 0)")

Upload your .db file:


Saving mad_before_phase1_5090_ragfix_01_.db to mad_before_phase1_5090_ragfix_01_.db
DB loaded at: /content/mad_before_phase1_5090_ragfix_01_.db
queries=50  claims=414  agent_outputs=0 (should be 0)


In [ ]:
# CELL 3: All config in one place — tune here only

MODEL_NAME   = "Qwen/Qwen2.5-14B-Instruct-AWQ"
VLLM_PORT    = 8001
VLLM_URL     = f"http://localhost:{VLLM_PORT}/v1"
SERVED_NAME  = "agents"

# Context window guards
MAX_CHUNK_CHARS   = 800   # per chunk text — ~200 tokens each, 5 chunks = ~1000 tokens
MAX_CLAIM_CHARS   = 300   # claim text safety cap
MAX_PEER_CHARS    = 500   # per peer reasoning in Round 1

# Temperatures per agent role
TEMPERATURES = {
    "agent_a": 0.3,   # Verifier — low temp, conservative
    "agent_b": 0.7,   # Adversarial Auditor — high temp, aggressive
    "agent_c": 0.5,   # Calibrator — moderate, balanced
}

# Concurrency: N claims debated simultaneously
# 3 agents × 4 claims = 12 concurrent requests → matches --max-num-seqs 12
CLAIM_CONCURRENCY = 4

MAX_TOKENS_OUT = 512   # max tokens per agent response
VLLM_TIMEOUT   = 120  # seconds per agent call

print("Config set.")
print(f"  Model: {MODEL_NAME}")
print(f"  Claim concurrency: {CLAIM_CONCURRENCY} claims × 3 agents = {CLAIM_CONCURRENCY*3} concurrent calls")

Config set.
  Model: Qwen/Qwen2.5-14B-Instruct-AWQ
  Claim concurrency: 4 claims × 3 agents = 12 concurrent calls


In [ ]:
# CELL 4: Launch vLLM server — takes 3-5 min to download + load model
import subprocess, threading

LOG_PATH = "/content/vllm_server.log"

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model",                  MODEL_NAME,
    "--quantization",           "awq",
    "--max-model-len",          "6144",
    "--gpu-memory-utilization", "0.88",
    "--max-num-seqs",           "12",
    "--enable-prefix-caching",
    "--port",                   str(VLLM_PORT),
    "--served-model-name",      SERVED_NAME,
    "--trust-remote-code",
    "--dtype",                  "float16",
]

print("Launching vLLM server...")
print("Command:", " ".join(vllm_cmd))

server_log = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    vllm_cmd,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
print(f"Server PID: {server_proc.pid}")
print(f"Logs: {LOG_PATH}")
print("Run CELL 5 to wait for it to be ready...")

Launching vLLM server...
Command: python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-14B-Instruct-AWQ --quantization awq --max-model-len 6144 --gpu-memory-utilization 0.88 --max-num-seqs 12 --enable-prefix-caching --port 8001 --served-model-name agents --trust-remote-code --dtype float16
Server PID: 9816
Logs: /content/vllm_server.log
Run CELL 5 to wait for it to be ready...


In [ ]:
# CELL 5: Poll until vLLM is ready (model load takes 2-4 min on A100)
import time, requests

def wait_for_vllm(url: str, timeout_secs: int = 300):
    print("Waiting for vLLM server", end="")
    for i in range(timeout_secs // 5):
        try:
            r = requests.get(f"{url}/models", timeout=3)
            if r.status_code == 200:
                models = r.json().get("data", [])
                print(f"\n✓ Server ready! Models: {[m['id'] for m in models]}")
                return True
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(5)
    print("\n✗ Server failed to start — check logs below")
    return False

ready = wait_for_vllm(VLLM_URL)

if not ready:
    # Print last 30 lines of log to diagnose
    !tail -30 /content/vllm_server.log

Waiting for vLLM server................................
✓ Server ready! Models: ['agents']


In [ ]:
# CELL 6: Data schemas — the contract between agents, DB, and judge

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from enum import Enum

class AgentVerdict(str, Enum):
    SUPPORTED     = "SUPPORTED"
    PARTIAL       = "PARTIAL"
    NOT_SUPPORTED = "NOT_SUPPORTED"
    IDK           = "IDK"

class AgentRole(str, Enum):
    AGENT_A = "agent_a"   # Verifier
    AGENT_B = "agent_b"   # Adversarial Auditor
    AGENT_C = "agent_c"   # Calibrator

class EvidenceCitation(BaseModel):
    chunk_id:       str
    relevant_quote: str

class AgentOutputFull(BaseModel):
    """Full output → written to SQLite. NEVER passed to peers or judge."""
    agent_role:          AgentRole
    round_num:           int
    verdict:             AgentVerdict
    reasoning:           str
    evidence_cited:      List[EvidenceCitation]
    confidence_internal: float = Field(ge=0.0, le=1.0)

class AgentOutputStripped(BaseModel):
    """Stripped output → safe to pass to peer agents. No confidence. No role attribution."""
    debater_label:  str        # "Debater 1", "Debater 2", "Debater 3"
    round_num:      int
    verdict:        AgentVerdict
    reasoning:      str        # may be truncated to MAX_PEER_CHARS
    evidence_cited: List[EvidenceCitation]

def strip_for_peer(output: AgentOutputFull, debater_label: str,
                   max_reasoning_chars: int = MAX_PEER_CHARS) -> AgentOutputStripped:
    """
    THE ONLY function that creates AgentOutputStripped.
    Removes confidence_internal and anonymizes agent identity.
    This is the bias control choke point — never bypass it.
    """
    return AgentOutputStripped(
        debater_label  = debater_label,
        round_num      = output.round_num,
        verdict        = output.verdict,
        reasoning      = output.reasoning[:max_reasoning_chars],
        evidence_cited = output.evidence_cited,
    )

# Verify bias control works
def _test_strip():
    full = AgentOutputFull(
        agent_role=AgentRole.AGENT_A, round_num=0,
        verdict=AgentVerdict.SUPPORTED, reasoning="test",
        evidence_cited=[], confidence_internal=0.85
    )
    stripped = strip_for_peer(full, "Debater 1")
    s = stripped.model_dump_json()
    assert "confidence" not in s.lower(), "LEAK: confidence in stripped output!"
    assert "agent_a"   not in s.lower(), "LEAK: agent role in stripped output!"
    assert "agent_b"   not in s.lower(), "LEAK: agent role in stripped output!"
    print("✓ strip_for_peer bias control test passed")

_test_strip()

✓ strip_for_peer bias control test passed


In [ ]:
# CELL 7: System prompts for all three agents

AGENT_A_SYSTEM = """You are a strict regulatory compliance verifier in a multi-agent debate.

YOUR ROLE: Determine whether the claim is supported by the retrieved evidence.
Approach the claim charitably — find the strongest case FOR the claim being correct.

RULES:
1. Read the claim and evidence carefully.
2. Find what the evidence DOES support about the claim.
3. Verdict must be one of: SUPPORTED / PARTIAL / NOT_SUPPORTED / IDK
   - SUPPORTED: evidence clearly backs the claim
   - PARTIAL: evidence partially supports but with gaps or qualifications
   - NOT_SUPPORTED: evidence contradicts or is silent on the claim
   - IDK: evidence is too ambiguous to decide
4. Cite specific chunk_ids for every point.
5. In Round 1, if other debaters raised valid challenges, acknowledge them honestly.
   Your goal is accuracy, not winning.
6. confidence_internal: your true certainty 0.0–1.0. Be honest, not optimistic.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Your detailed analysis referencing specific evidence...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "..."}],
    "confidence_internal": 0.0
}"""

AGENT_B_SYSTEM = """You are an adversarial auditor in a multi-agent regulatory compliance debate.

YOUR ROLE: Challenge the claim. Find what is WRONG, INCOMPLETE, or MISLEADING
given only the retrieved evidence chunks.

RULES:
1. Read the claim and evidence carefully.
2. Look specifically for:
   - Facts in the claim not supported by the evidence
   - Overly broad or narrow scope (claim generalizes beyond what evidence shows)
   - Missing exceptions, conditions, or carve-outs present in the evidence
   - Regulatory nuance the claim flattens or ignores
3. Verdict must be one of: SUPPORTED / PARTIAL / NOT_SUPPORTED / IDK
4. Cite specific chunk_ids for every challenge.
5. If the claim is genuinely well-supported by evidence, say so — your credibility requires accuracy.
6. In Round 1, engage directly with other debaters. Push back if their argument is weak.
7. confidence_internal: your true certainty 0.0–1.0.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Your detailed challenge referencing specific evidence...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "..."}],
    "confidence_internal": 0.0
}"""

AGENT_C_SYSTEM = """You are a neutral evidence calibrator in a multi-agent regulatory compliance debate.

YOUR ROLE: You have NO prior stance. Weigh the evidence honestly on BOTH sides
and report what the retrieved chunks actually support.

RULES:
1. Read the claim and evidence carefully with zero agenda.
2. Explicitly consider BOTH directions:
   - What in the evidence supports the claim?
   - What in the evidence undermines or qualifies the claim?
3. Verdict must be one of: SUPPORTED / PARTIAL / NOT_SUPPORTED / IDK
   - PARTIAL is the honest answer when evidence cuts both ways.
   - Do not manufacture false balance — if evidence clearly favors one side, say so.
4. Cite specific chunk_ids for BOTH supporting and undermining points.
5. In Round 1, after seeing other debaters' positions:
   - Identify which argument was better grounded in the evidence
   - Update your position if warranted, or explain why neither convinced you
6. confidence_internal: calibrated certainty 0.0–1.0.
   0.5 means genuinely uncertain — not a default hedge.

OUTPUT FORMAT: Valid JSON only. No markdown. No text outside the JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Your balanced analysis with evidence for and against...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "..."}],
    "confidence_internal": 0.0
}"""

SYSTEM_PROMPTS = {
    "agent_a": AGENT_A_SYSTEM,
    "agent_b": AGENT_B_SYSTEM,
    "agent_c": AGENT_C_SYSTEM,
}

print("✓ System prompts defined")
print(f"  Agent A prompt: {len(AGENT_A_SYSTEM)} chars")
print(f"  Agent B prompt: {len(AGENT_B_SYSTEM)} chars")
print(f"  Agent C prompt: {len(AGENT_C_SYSTEM)} chars")

✓ System prompts defined
  Agent A prompt: 1209 chars
  Agent B prompt: 1212 chars
  Agent C prompt: 1342 chars


In [ ]:
# CELL 8: SQLite read/write helpers

import sqlite3, json, threading, uuid
from datetime import datetime

db_lock = threading.Lock()  # Prevent concurrent writes corrupting DB

def new_id() -> str:
    return str(uuid.uuid4())

# ── READERS ──────────────────────────────────────────────────────────────────

def load_all_queries() -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT * FROM queries ORDER BY query_id").fetchall()
    conn.close()
    result = []
    for r in rows:
        d = dict(r)
        d["rag_chunks"] = json.loads(d["rag_chunks"])
        d["rag_chunk_ids"] = json.loads(d["rag_chunk_ids"])
        result.append(d)
    return result

def load_claims_for_query(query_id: str) -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT * FROM claims WHERE query_id=? ORDER BY claim_index",
        (query_id,)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def query_already_done(query_id: str) -> bool:
    """Skip queries where all claims already have agent outputs."""
    conn = sqlite3.connect(DB_PATH)
    claim_ids = [r[0] for r in conn.execute(
        "SELECT claim_id FROM claims WHERE query_id=?", (query_id,)
    ).fetchall()]
    conn.close()
    if not claim_ids:
        return False
    conn = sqlite3.connect(DB_PATH)
    done = conn.execute(
        f"SELECT COUNT(DISTINCT claim_id) FROM agent_outputs "
        f"WHERE claim_id IN ({','.join('?'*len(claim_ids))}) AND round_num=1",
        claim_ids
    ).fetchone()[0]
    conn.close()
    return done == len(claim_ids)

# ── WRITERS ──────────────────────────────────────────────────────────────────

def write_agent_output(
    claim_id: str, agent_role: str, round_num: int,
    verdict: str, reasoning: str, evidence_cited: list,
    confidence_internal: float, raw_response: str,
    latency_ms: int, tokens_in: int, tokens_out: int
):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_outputs
            (output_id, claim_id, agent_role, round_num, verdict, reasoning,
             evidence_cited, confidence_internal, raw_response,
             latency_ms, tokens_in, tokens_out, timestamp)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP)
        """, (
            new_id(), claim_id, agent_role, round_num,
            verdict, reasoning, json.dumps(evidence_cited),
            confidence_internal, raw_response,
            latency_ms, tokens_in, tokens_out
        ))
        conn.commit()
        conn.close()

def write_agent_delta(
    claim_id: str, agent_role: str,
    conf_r0: float, conf_r1: float,
    verdict_r0: str, verdict_r1: str
):
    delta = conf_r1 - conf_r0
    verdict_changed = verdict_r0 != verdict_r1
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_deltas
            (delta_id, claim_id, agent_role,
             confidence_r0, confidence_r1, delta,
             verdict_r0, verdict_r1, verdict_changed)
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (
            new_id(), claim_id, agent_role,
            conf_r0, conf_r1, delta,
            verdict_r0, verdict_r1, verdict_changed
        ))
        conn.commit()
        conn.close()

print("✓ DB helpers ready")
# Quick test
queries = load_all_queries()
print(f"  Loaded {len(queries)} queries")
claims_q1 = load_claims_for_query("q_001")
print(f"  q_001 has {len(claims_q1)} claims")

✓ DB helpers ready
  Loaded 50 queries
  q_001 has 11 claims


In [ ]:
# CELL 9: Async HTTP client for vLLM + robust JSON parser

import aiohttp, asyncio, time, re, json

def parse_agent_json(raw: str) -> dict:
    """
    Robust parser — handles: raw JSON, markdown-fenced JSON, JSON embedded in text.
    Always returns a valid dict with required keys, never raises.
    """
    raw = raw.strip()

    # Attempt 1: direct parse
    try:
        return json.loads(raw)
    except Exception:
        pass

    # Attempt 2: strip markdown code fence
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass

    # Attempt 3: find first { ... } block (greedy)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    # Attempt 4: partial field extraction via regex
    verdict_match = re.search(r'"verdict"\s*:\s*"([^"]+)"', raw)
    conf_match    = re.search(r'"confidence_internal"\s*:\s*([0-9.]+)', raw)
    reason_match  = re.search(r'"reasoning"\s*:\s*"([^"]+)"', raw)

    return {
        "verdict":             verdict_match.group(1) if verdict_match else "IDK",
        "reasoning":           reason_match.group(1) if reason_match else f"[PARSE_FAILED] {raw[:300]}",
        "evidence_cited":      [],
        "confidence_internal": float(conf_match.group(1)) if conf_match else 0.5,
    }

def normalize_verdict(v: str) -> str:
    v = v.upper().strip()
    mapping = {
        "SUPPORT": "SUPPORTED", "SUPPORTED": "SUPPORTED",
        "NOT_SUPPORTED": "NOT_SUPPORTED", "NOTSUPPORTED": "NOT_SUPPORTED",
        "NOT SUPPORTED": "NOT_SUPPORTED", "UNSUPPORTED": "NOT_SUPPORTED",
        "PARTIAL": "PARTIAL", "PARTIALLY": "PARTIAL",
        "IDK": "IDK", "UNKNOWN": "IDK", "INSUFFICIENT": "IDK",
    }
    return mapping.get(v, "IDK")

async def call_agent_async(
    session:      aiohttp.ClientSession,
    system_prompt: str,
    user_prompt:   str,
    temperature:   float,
) -> tuple[dict, str, int, int, int]:
    """
    Returns: (parsed_dict, raw_text, latency_ms, tokens_in, tokens_out)
    Never raises — returns IDK fallback on any error.
    """
    payload = {
        "model":       SERVED_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens":  MAX_TOKENS_OUT,
        "stream":      False,
    }

    start = time.time()
    try:
        async with session.post(
            f"{VLLM_URL}/chat/completions",
            json=payload,
            timeout=aiohttp.ClientTimeout(total=VLLM_TIMEOUT),
        ) as resp:
            resp.raise_for_status()
            data       = await resp.json()
            latency_ms = int((time.time() - start) * 1000)
            raw_text   = data["choices"][0]["message"]["content"]
            tokens_in  = data["usage"]["prompt_tokens"]
            tokens_out = data["usage"]["completion_tokens"]
    except Exception as e:
        latency_ms = int((time.time() - start) * 1000)
        raw_text   = f"[CALL_ERROR] {str(e)}"
        tokens_in  = tokens_out = 0

    parsed = parse_agent_json(raw_text)
    parsed["verdict"] = normalize_verdict(parsed.get("verdict", "IDK"))

    # Clamp confidence to [0.0, 1.0]
    c = parsed.get("confidence_internal", 0.5)
    parsed["confidence_internal"] = max(0.0, min(1.0, float(c)))

    return parsed, raw_text, latency_ms, tokens_in, tokens_out

print("✓ vLLM client ready")

# Quick smoke test (optional — uncomment to test)
# async def smoke_test():
#     async with aiohttp.ClientSession() as session:
#         parsed, raw, lat, ti, to = await call_agent_async(
#             session, "You are a helpful assistant.", "Say hello in JSON: {\"msg\": \"...\"}", 0.3
#         )
#         print(f"Smoke test → {parsed} | latency={lat}ms | tokens={ti}+{to}")
# asyncio.run(smoke_test())

✓ vLLM client ready


In [ ]:
# CELL 10: Build user prompts for Round 0 and Round 1

def format_chunks(chunks: list, max_chars: int = MAX_CHUNK_CHARS) -> str:
    """Format RAG chunks for agent prompts. Truncates each chunk text."""
    parts = []
    for i, c in enumerate(chunks):
        text = c["text"][:max_chars]
        if len(c["text"]) > max_chars:
            text += "... [truncated]"
        parts.append(
            f"[Chunk {i+1} | ID: {c['chunk_id']} | Source: {c['source_file']}]\n{text}"
        )
    return "\n\n".join(parts)

def build_round0_prompt(query: str, claim_text: str, chunks: list) -> str:
    claim_text = claim_text[:MAX_CLAIM_CHARS]
    return f"""USER QUERY: {query}

CLAIM TO VERIFY:
{claim_text}

RETRIEVED EVIDENCE:
{format_chunks(chunks)}

Provide your independent verdict on this claim."""

def build_round1_prompt(
    query: str,
    claim_text: str,
    chunks: list,
    peer1: AgentOutputStripped,
    peer2: AgentOutputStripped,
) -> str:
    claim_text = claim_text[:MAX_CLAIM_CHARS]

    def fmt_peer(p: AgentOutputStripped) -> str:
        evidence_str = ", ".join(e.chunk_id for e in p.evidence_cited[:3]) or "none"
        return (
            f"Verdict: {p.verdict}\n"
            f"Reasoning: {p.reasoning[:MAX_PEER_CHARS]}\n"
            f"Evidence cited: {evidence_str}"
        )

    return f"""USER QUERY: {query}

CLAIM TO VERIFY:
{claim_text}

RETRIEVED EVIDENCE:
{format_chunks(chunks)}

OTHER DEBATERS' POSITIONS:

[{peer1.debater_label}]
{fmt_peer(peer1)}

[{peer2.debater_label}]
{fmt_peer(peer2)}

Review the positions above carefully. You may update or maintain your Round 0 verdict.
If a debater made a valid point supported by evidence, acknowledge it.
If their argument is weak or unsupported, push back with evidence.
Provide your final verdict."""

print("✓ Prompt builders ready")

# Show worst-case token estimate
import json
sample_q = queries[22]  # q_023 — largest chunks
sample_c = load_claims_for_query(sample_q["query_id"])[0]
r0_prompt = build_round0_prompt(sample_q["user_query"], sample_c["claim_text"], sample_q["rag_chunks"])
print(f"Worst-case R0 prompt: {len(r0_prompt)} chars ≈ {len(r0_prompt)//4} tokens")
print(f"System prompt: ≈ {len(AGENT_A_SYSTEM)//4} tokens")
print(f"Estimated total input: ≈ {(len(r0_prompt)+len(AGENT_A_SYSTEM))//4} tokens (limit=6144)")

✓ Prompt builders ready
Worst-case R0 prompt: 5156 chars ≈ 1289 tokens
System prompt: ≈ 302 tokens
Estimated total input: ≈ 1591 tokens (limit=6144)


In [ ]:
# CELL 11: Round 0 — all three agents independently assess each claim

import asyncio, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("MAD")

AGENT_ROLES   = ["agent_a", "agent_b", "agent_c"]
DEBATER_LABEL = {"agent_a": "Debater 1", "agent_b": "Debater 2", "agent_c": "Debater 3"}

async def debate_claim_round0(
    session:    aiohttp.ClientSession,
    query:      dict,
    claim:      dict,
) -> dict:
    """
    Fire all 3 agents simultaneously on one claim.
    Returns: {agent_role: AgentOutputFull}
    """
    tasks = {}
    for role in AGENT_ROLES:
        prompt = build_round0_prompt(
            query["user_query"],
            claim["claim_text"],
            query["rag_chunks"],
        )
        tasks[role] = asyncio.create_task(
            call_agent_async(
                session,
                SYSTEM_PROMPTS[role],
                prompt,
                TEMPERATURES[role],
            )
        )

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to = await task

        # Build full output object
        evidence_cited = [
            EvidenceCitation(**e) if isinstance(e, dict) else e
            for e in parsed.get("evidence_cited", [])
        ]
        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=0,
            verdict=AgentVerdict(parsed["verdict"]),
            reasoning=parsed.get("reasoning", ""),
            evidence_cited=evidence_cited,
            confidence_internal=parsed["confidence_internal"],
        )
        results[role] = output

        # Write to DB immediately
        write_agent_output(
            claim_id=claim["claim_id"],
            agent_role=role, round_num=0,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=[e.model_dump() for e in output.evidence_cited],
            confidence_internal=output.confidence_internal,
            raw_response=raw,
            latency_ms=lat, tokens_in=ti, tokens_out=to,
        )

        logger.info(
            f"R0 | {claim['claim_id'][:8]} | {role} | "
            f"{output.verdict.value} | conf={output.confidence_internal:.2f} | {lat}ms"
        )

    return results

print("✓ Round 0 debate function ready")

✓ Round 0 debate function ready


In [ ]:
# CELL 12: Round 1 — each agent sees the other TWO agents' stripped R0 outputs

async def debate_claim_round1(
    session:    aiohttp.ClientSession,
    query:      dict,
    claim:      dict,
    r0_outputs: dict,   # {agent_role: AgentOutputFull}
) -> dict:
    """
    Round 1: each agent sees the other two agents' R0 (stripped — no confidence, no attribution).
    Returns: {agent_role: AgentOutputFull}
    """
    # Build stripped peer views for every agent
    # agent_a sees stripped(agent_b) + stripped(agent_c)
    # agent_b sees stripped(agent_a) + stripped(agent_c)
    # agent_c sees stripped(agent_a) + stripped(agent_b)

    stripped = {
        role: strip_for_peer(output, DEBATER_LABEL[role])
        for role, output in r0_outputs.items()
    }

    other_roles = {
        "agent_a": ("agent_b", "agent_c"),
        "agent_b": ("agent_a", "agent_c"),
        "agent_c": ("agent_a", "agent_b"),
    }

    tasks = {}
    for role in AGENT_ROLES:
        p1_role, p2_role = other_roles[role]
        prompt = build_round1_prompt(
            query["user_query"],
            claim["claim_text"],
            query["rag_chunks"],
            peer1=stripped[p1_role],
            peer2=stripped[p2_role],
        )
        tasks[role] = asyncio.create_task(
            call_agent_async(
                session,
                SYSTEM_PROMPTS[role],
                prompt,
                TEMPERATURES[role],
            )
        )

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to = await task

        evidence_cited = [
            EvidenceCitation(**e) if isinstance(e, dict) else e
            for e in parsed.get("evidence_cited", [])
        ]
        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=1,
            verdict=AgentVerdict(parsed["verdict"]),
            reasoning=parsed.get("reasoning", ""),
            evidence_cited=evidence_cited,
            confidence_internal=parsed["confidence_internal"],
        )
        results[role] = output

        # Write Round 1 output to DB
        write_agent_output(
            claim_id=claim["claim_id"],
            agent_role=role, round_num=1,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=[e.model_dump() for e in output.evidence_cited],
            confidence_internal=output.confidence_internal,
            raw_response=raw,
            latency_ms=lat, tokens_in=ti, tokens_out=to,
        )

        # Write delta (R0 → R1)
        r0 = r0_outputs[role]
        write_agent_delta(
            claim_id=claim["claim_id"],
            agent_role=role,
            conf_r0=r0.confidence_internal,
            conf_r1=output.confidence_internal,
            verdict_r0=r0.verdict.value,
            verdict_r1=output.verdict.value,
        )

        logger.info(
            f"R1 | {claim['claim_id'][:8]} | {role} | "
            f"{r0.verdict.value}→{output.verdict.value} | "
            f"conf {r0.confidence_internal:.2f}→{output.confidence_internal:.2f} | "
            f"Δ={output.confidence_internal - r0.confidence_internal:+.2f} | {lat}ms"
        )

    return results

print("✓ Round 1 debate function ready")

✓ Round 1 debate function ready


In [ ]:
# CELL 13: Orchestrate full debate for one query (all its claims)

async def debate_query(
    session:    aiohttp.ClientSession,
    query:      dict,
    sem:        asyncio.Semaphore,
) -> dict:
    """
    Run full 2-round 3-agent debate for every claim in this query.
    Semaphore limits how many claims are debated concurrently.
    Returns: summary stats for this query.
    """
    claims = load_claims_for_query(query["query_id"])
    if not claims:
        logger.warning(f"No claims for {query['query_id']}, skipping")
        return {"query_id": query["query_id"], "claims": 0}

    async def debate_one_claim(claim: dict):
        async with sem:
            logger.info(f"→ Claim {claim['claim_index']} | {claim['claim_id'][:8]} | {claim['claim_text'][:60]}...")
            # Round 0
            r0 = await debate_claim_round0(session, query, claim)
            # Round 1
            r1 = await debate_claim_round1(session, query, claim, r0)
            return r0, r1

    tasks = [debate_one_claim(c) for c in claims]
    all_results = await asyncio.gather(*tasks, return_exceptions=True)

    errors = [r for r in all_results if isinstance(r, Exception)]
    if errors:
        logger.error(f"Query {query['query_id']} had {len(errors)} claim errors: {errors[:2]}")

    return {
        "query_id": query["query_id"],
        "claims":   len(claims),
        "errors":   len(errors),
    }

print("✓ Per-query orchestrator ready")

✓ Per-query orchestrator ready


In [ ]:
# CELL 14: Main run — all 50 queries
# Skips queries that already have complete agent_outputs (safe to re-run)

async def run_all_queries():
    queries = load_all_queries()
    sem     = asyncio.Semaphore(CLAIM_CONCURRENCY)

    total_claims = sum(len(load_claims_for_query(q["query_id"])) for q in queries)
    print(f"Starting MAD debate: {len(queries)} queries, {total_claims} claims")
    print(f"3 agents × 2 rounds × {total_claims} claims = {total_claims*6} LLM calls total")
    print(f"Claim concurrency: {CLAIM_CONCURRENCY} (= {CLAIM_CONCURRENCY*3} concurrent agent calls)\n")

    start_time = time.time()
    summaries  = []

    async with aiohttp.ClientSession() as session:
        for i, query in enumerate(queries):
            qid = query["query_id"]

            if query_already_done(qid):
                print(f"[{i+1:02d}/50] {qid} — already done, skipping")
                continue

            claims = load_claims_for_query(qid)
            print(f"[{i+1:02d}/50] {qid} | {len(claims)} claims | {query['user_query'][:60]}...")

            try:
                summary = await debate_query(session, query, sem)
                summaries.append(summary)
            except Exception as e:
                logger.error(f"Query {qid} failed: {e}")
                summaries.append({"query_id": qid, "claims": 0, "errors": 1})

            # Progress
            elapsed = time.time() - start_time
            done_q  = i + 1
            rate    = done_q / elapsed
            eta     = (len(queries) - done_q) / rate if rate > 0 else 0
            print(f"    elapsed={elapsed/60:.1f}min | ETA={eta/60:.1f}min\n")

    total_elapsed = time.time() - start_time
    print(f"\n✓ All queries done in {total_elapsed/60:.1f} minutes")
    print(f"  Total summaries: {len(summaries)}")
    return summaries

# RUN IT
summaries = asyncio.run(run_all_queries())

Starting MAD debate: 50 queries, 414 claims
3 agents × 2 rounds × 414 claims = 2484 LLM calls total
Claim concurrency: 4 (= 12 concurrent agent calls)

[01/50] q_001 | 11 claims | What is the relationship between self-reported physical acti...
    elapsed=3.6min | ETA=175.0min

[02/50] q_002 | 2 claims | What must be disregarded in the proceeding according to the ...
    elapsed=5.0min | ETA=120.9min

[03/50] q_003 | 5 claims | What does 'contrary' mean in the context of comparing State ...
    elapsed=7.1min | ETA=111.7min

[04/50] q_004 | 10 claims | What must a covered entity do when making routine and recurr...
    elapsed=10.5min | ETA=120.2min

[05/50] q_005 | 9 claims | What is the requirement for using the Employer Identifier in...
    elapsed=13.3min | ETA=119.6min

[06/50] q_006 | 10 claims | What are the potential effects of Dronedarone on heart funct...
    elapsed=16.5min | ETA=121.3min

[07/50] q_007 | 6 claims | What is the recommended dietary protein intake for individu

ERROR:MAD:Query q_023 had 1 claim errors: [1 validation error for EvidenceCitation
relevant_quote
  Field required [type=missing, input_value={'chunk_id': 'FDA_Predete...l_Plan_fo_ Yöntem yok'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing]


    elapsed=59.8min | ETA=70.2min

[24/50] q_024 | 13 claims | What steps must a competent authority take upon receiving re...
    elapsed=64.7min | ETA=70.1min

[25/50] q_025 | 10 claims | What must a covered entity do if restricted protected health...
    elapsed=67.2min | ETA=67.2min

[26/50] q_026 | 4 claims | What plasma glucose values indicate a diagnosis of gestation...
    elapsed=68.8min | ETA=63.5min

[27/50] q_027 | 9 claims | What is the recommended action for individuals with prediabe...
    elapsed=72.2min | ETA=61.5min

[28/50] q_028 | 7 claims | What is the responsibility of the Federal Trade Commission u...
    elapsed=74.6min | ETA=58.6min

[29/50] q_029 | 3 claims | What are the affiliations of Carmelo A. Milano as listed in ...
    elapsed=75.5min | ETA=54.7min

[30/50] q_030 | 11 claims | What information must be communicated to outpatient health c...
    elapsed=78.7min | ETA=52.5min

[31/50] q_031 | 10 claims | What is the requirement for the CE marking when a no

In [ ]:
# CELL 15: Verify results and inspect debate quality

import sqlite3, json

conn = sqlite3.connect(DB_PATH)

# Row counts
print("=== TABLE COUNTS ===")
for table in ["queries", "claims", "agent_outputs", "agent_deltas"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

# Verdict distribution per agent
print("\n=== VERDICT DISTRIBUTION (Round 1) ===")
rows = conn.execute("""
    SELECT agent_role, verdict, COUNT(*) as n
    FROM agent_outputs
    WHERE round_num = 1
    GROUP BY agent_role, verdict
    ORDER BY agent_role, verdict
""").fetchall()
for r in rows:
    print(f"  {r[0]} | {r[1]:<15} | {r[2]}")

# Average confidence per agent per round
print("\n=== AVG CONFIDENCE PER AGENT PER ROUND ===")
rows = conn.execute("""
    SELECT agent_role, round_num, ROUND(AVG(confidence_internal),3) as avg_conf
    FROM agent_outputs
    GROUP BY agent_role, round_num
    ORDER BY agent_role, round_num
""").fetchall()
for r in rows:
    print(f"  {r[0]} R{r[1]} | avg_conf={r[2]}")

# Large confidence shifts (potential GRPO gold)
print("\n=== TOP 10 LARGEST CONFIDENCE DELTAS ===")
rows = conn.execute("""
    SELECT d.agent_role, d.claim_id, d.delta,
           d.verdict_r0, d.verdict_r1, d.verdict_changed,
           c.claim_text
    FROM agent_deltas d
    JOIN claims c USING (claim_id)
    ORDER BY ABS(d.delta) DESC
    LIMIT 10
""").fetchall()
for r in rows:
    changed = "✓ FLIPPED" if r[5] else ""
    print(f"  {r[0]} | Δ={r[2]:+.2f} | {r[3]}→{r[4]} {changed}")
    print(f"    claim: {r[6][:80]}")

# Disagreement between agents (judge-needed cases)
print("\n=== CLAIMS WHERE ALL 3 AGENTS DISAGREE (R1) ===")
rows = conn.execute("""
    SELECT claim_id, GROUP_CONCAT(agent_role||':'||verdict, ' | ') as positions
    FROM agent_outputs
    WHERE round_num = 1
    GROUP BY claim_id
    HAVING COUNT(DISTINCT verdict) = 3
    LIMIT 10
""").fetchall()
print(f"  Total 3-way disagreements: {len(rows)}")
for r in rows[:5]:
    print(f"  {r[0][:12]} | {r[1]}")

conn.close()

=== TABLE COUNTS ===
  queries: 50 rows
  claims: 414 rows
  agent_outputs: 2479 rows
  agent_deltas: 1239 rows

=== VERDICT DISTRIBUTION (Round 1) ===
  agent_a | IDK             | 4
  agent_a | NOT_SUPPORTED   | 109
  agent_a | PARTIAL         | 291
  agent_a | SUPPORTED       | 9
  agent_b | IDK             | 7
  agent_b | NOT_SUPPORTED   | 114
  agent_b | PARTIAL         | 286
  agent_b | SUPPORTED       | 6
  agent_c | NOT_SUPPORTED   | 124
  agent_c | PARTIAL         | 278
  agent_c | SUPPORTED       | 11

=== AVG CONFIDENCE PER AGENT PER ROUND ===
  agent_a R0 | avg_conf=0.751
  agent_a R1 | avg_conf=0.781
  agent_b R0 | avg_conf=0.872
  agent_b R1 | avg_conf=0.855
  agent_c R0 | avg_conf=0.762
  agent_c R1 | avg_conf=0.789

=== TOP 10 LARGEST CONFIDENCE DELTAS ===
  agent_b | Δ=+1.00 | NOT_SUPPORTED→NOT_SUPPORTED 
    claim: These values represent general guidelines but may not apply to all systems or me
  agent_b | Δ=+1.00 | NOT_SUPPORTED→NOT_SUPPORTED 
    claim: The ETSI Gro

In [ ]:
# CELL 16: Download the updated DB back to your machine

from google.colab import files

print("Downloading updated DB...")
files.download(DB_PATH)
print("Done. DB now contains agent_outputs + agent_deltas for all claims.")
print("Next step: Stage 2 — run the Judge on this DB.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. DB now contains agent_outputs + agent_deltas for all claims.
Next step: Stage 2 — run the Judge on this DB.


In [ ]:
# STEP 1: Check what we have
conn = sqlite3.connect(DB_PATH)

ao = conn.execute("SELECT COUNT(*) FROM agent_outputs").fetchone()[0]
ad = conn.execute("SELECT COUNT(*) FROM agent_deltas").fetchone()[0]
print(f"agent_outputs: {ao}  (expected ~2484)")
print(f"agent_deltas:  {ad}  (expected ~1242)")

# Find any missing claims
missing = conn.execute("""
    SELECT c.claim_id, c.query_id, c.claim_text
    FROM claims c
    WHERE c.claim_id NOT IN (
        SELECT DISTINCT claim_id FROM agent_outputs WHERE round_num = 1
    )
""").fetchall()
print(f"\nMissing R1 outputs: {len(missing)} claims")
for m in missing:
    print(f"  {m[1]} | {m[0][:12]} | {m[2][:70]}")
conn.close()

agent_outputs: 2479  (expected ~2484)
agent_deltas:  1239  (expected ~1242)

Missing R1 outputs: 1 claims
  q_023 | 81e874e9-1e6 | For Agilent 1200 Series, the RSD for peak area measurements should be 
